In [68]:
import sys
from search import hybrid_query,bm25_search,Embed_search
import ollama
import pandas as pd

In [2]:
hybrid_query("inflation transitory")

,chunk_id,score_embed,rank_embed,score_bm25,rank_bm25,rrf,date,text
0,2021-09-22_p2_c1,0.427782,11.0,4.472144,2.0,0.030214,2021-09-22,With progress on vaccinations and strong polic...
1,2021-07-28_p2_c1,0.419378,13.0,4.714713,1.0,0.030092,2021-07-28,With progress on vaccinations and strong polic...
2,2024-05-01_p2_c1,0.442340,5.0,1.463931,8.0,0.030090,2024-05-01,The Committee seeks to achieve maximum employm...
3,2024-01-31_p2_c1,0.436827,7.0,1.492150,6.0,0.030077,2024-01-31,The Committee seeks to achieve maximum employm...
4,2024-06-12_p2_c1,0.442340,6.0,1.463931,9.0,0.029644,2024-06-12,The Committee seeks to achieve maximum employm...


In [51]:
def answer(question,k=5):
    results=hybrid_query(question,k)[["date","text"]]
    blocks=[]
    for _,r in results.iterrows():
       blocks.append(f"[{r['date']}] {r['text']}")
    context="\n\n".join(blocks)
    messages = [
       {"role": "system", "content": "You answer questions about Federal Reserve FOMC statements. "
                                     "Use only the statements provided. Cite the date in brackets after each claim, like [2022-06-15]. "
                                     "If the statements don't contain the answer, say so. Answer in plain text, no tables."},
       {"role": "user", "content": f"Statements:\n{context}\n\nQuestion: {question}"},
   ]
    response = ollama.chat(
    model=MODEL,
    messages=messages,
    )
    return response["message"]["content"]

In [52]:
print(answer("Did the Fed think inflation was transitory?"))

Yes. The Fed characterized the rise in inflation as largely reflecting transitory factors in several statements: [2021-09-22], [2021-06-16], [2021-11-03], and [2021-07-28].


In [53]:
print(answer("When did the Fed raise rates by 75 basis points?"))

The statements you provided do not mention any 75‑basis‑point rate increase.  All of the rate‑change notices refer to 1/4‑percentage‑point moves (i.e., 25 basis points).  Therefore, based on the available statements, I cannot identify a 75‑basis‑point raise.


In [54]:
print(answer("What did the Fed say about bitcoin?"))

The statements provided do not contain any mention of bitcoin.


In [56]:
print(answer("What did the Fed say about 2026 inflation"))

The Fed said that inflation remains elevated in 2026 and that accumulating more information will help it determine the policy stance needed to bring inflation back to its 2 percent goal, with the goal of delivering price stability. [2026-10-28]


In [57]:
eval_set = [
    {"question": "Did the Fed think inflation was transitory?",
     "expected": ["2021-06-16", "2021-07-28", "2021-09-22", "2021-11-03"]},

    {"question": "When did the Fed start reducing its balance sheet?",
     "expected": ["2022-05-04"]},

    {"question": "When did the Fed stop reducing its balance sheet?",
     "expected": ["2025-10-29"]},

    {"question": "When did the Fed raise rates by 75 basis points?",
     "expected": ["2022-06-15", "2022-07-27", "2022-09-21", "2022-11-02"]},

    {"question": "When did the Fed first raise rates after the pandemic?",
     "expected": ["2022-03-16"]},

    {"question": "Who dissented at the March 2022 meeting?",
     "expected": ["2022-03-16"]},

    {"question": "When did the Fed first cut rates after the 2022-2023 hikes?",
     "expected": ["2024-09-18"]},

    {"question": "How did the Fed describe the impact of COVID-19 on the economy?",
     "expected": ["2021-01-27", "2021-03-17", "2021-04-28"]},

    {"question": "When did the Fed change its long-run policy framework?",
     "expected": ["2025-08-22"]},

    {"question": "When did the Fed reduce the pace of its asset purchases?",
     "expected": ["2021-11-03", "2021-12-15"]},

    {"question": "What did the Fed say about bitcoin?",
     "expected": []},
]

In [65]:
def top_dates(method, question, k=5):
    if method == "bm25":
        results = bm25_search(question, k=k)          # list of (chunk_id, score)
        ids = [cid for cid, _ in results]
    elif method == "embed":
        results = Embed_search(question, k=k)
        ids = [cid for cid, _ in results]
    else:
        ids = list(hybrid_query(question, k=k)["chunk_id"])
    return [cid[:10] for cid in ids]

In [69]:
rows = []
for item in eval_set:
    if not item["expected"]:          # skip questions with no answer, like bitcoin
        continue
    row = {"question": item["question"]}
    for method in ["bm25", "embed", "hybrid"]:
        dates = top_dates(method, item["question"])
        row[method] = any(d in item["expected"] for d in dates)
    rows.append(row)

results_df = pd.DataFrame(rows)
results_df

,question,bm25,embed,hybrid
0,Did the Fed think inflation was transitory?,True,False,True
1,When did the Fed start reducing its balance sh...,False,True,True
2,When did the Fed stop reducing its balance sheet?,False,False,False
3,When did the Fed raise rates by 75 basis points?,False,True,True
4,When did the Fed first raise rates after the p...,False,False,False
5,Who dissented at the March 2022 meeting?,True,False,False
6,When did the Fed first cut rates after the 202...,False,False,False
7,How did the Fed describe the impact of COVID-1...,False,True,True
8,When did the Fed change its long-run policy fr...,True,True,True
9,When did the Fed reduce the pace of its asset ...,True,True,True
